# 🏢 Getting Started: Gemma 4 Developer Agent with HADL Dual-Loop Architecture
### Autonomous Cognitive Software Engineering on Gemma-4-31B | Kaggle Competition

This notebook produces the official, verified **`submission.zip`** for the **Gemma 4 Developer Agent Competition** using the **HADL Dual-Loop Architecture** calibrated with empirical findings from **Budget-Fit (0.18 LB)** and **Measure-Before-You-Tune (129 paired runs)**.

### Key Architectural Highlights:
1. **Calibrated Single-Agent Dual-Process**: Direct System 1 fast execution + System 2 latent deliberation without sub-agent turn lag.
2. **Harness Timeout Boost (`timeout_seconds: 240`)**: Eliminates grading test suite timeouts (e.g. `requests` test suite ~100 s).
3. **Budget Calibration (`8 min, 28 tool calls`)**: Gives Gemma sufficient time to resolve complex bugs while enforcing the 12-call edit deadline.
4. **Cognitive Triad (`/goal` ➔ `/plan` ➔ `/boost` ➔ `/goal`)**: Disciplined sequential execution where each cognitive phase reinforces the next without conflict.
5. **Pre-Submission Invariant Guard**: Strict Popperian firewall ensuring `tests/` remain untouched and scratch files are cleaned before submission.


## 1. Environment Configuration and Safe Initialization

We configure environment variables, locate competition datasets dynamically, and handle offline wheelhouse packages safely.


In [ ]:
import os
import sys
import glob
import json
import shutil
import zipfile
import hashlib
import platform
import subprocess
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'

# Working directory setup
WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
AGENT_DIR = WORKING_DIR / 'gemma4_xalpha_agent'
ZIP_OUTPUT_PATH = WORKING_DIR / 'submission.zip'

# Locate competition data directory dynamically across all potential Kaggle input paths
DATA_DIR_CANDIDATES = [
    Path('/kaggle/input/competitions/gemma-4-developer-agent'),
    Path('/kaggle/input/gemma-4-developer-agent'),
    Path('competition'),
    Path('.'),
]
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if (p / 'tasks.jsonl').exists()), None)
if DATA_DIR is None:
    DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.exists()), Path('.'))

print(f'[+] Environment: Python {platform.python_version()} on {platform.system()}')
print(f'[+] Data directory resolved: {DATA_DIR}')
print(f'[+] Working directory: {WORKING_DIR}')


## 2. Materialize Agent X-Alpha Dual-Loop Hierarchy

We construct the clean 5-file ADK agent bundle (`swe_coder` root agent + `code_analyzer` sub-agent).


In [ ]:
# Clean and create agent directory
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR, ignore_errors=True)
AGENT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = """You are an expert autonomous software engineer assigned to resolve an issue in a repository efficiently and decisively.

## Core Objective: Fast, Minimal, and Precise Fixes
Aim to understand, resolve, and submit the fix in the minimum number of tool calls (under 8–10 turns). Move directly from the problem statement to the relevant files, apply the solution, verify with a targeted test, and submit.

## Workflow

### 1. Identify Target Files Immediately
- Extract filenames, functions, classes, CLI subcommands, or error messages directly from the problem statement. Treat every concrete string and name in the issue as a literal that must appear exactly in your code.
- **Repository Layout**:
  - For `requests` issues: Source code is located under `src/requests/` (e.g. `src/requests/models.py`), NOT `requests/`.
  - For `httpx` issues: Source code is under `src/ahttpx/`.
  - For `fastapi` issues: Implementation is under `fastapi/`. If the issue involves tutorials, documentation examples, or license metadata, the target file is under `docs_src/**/tutorial*.py`.
  - For `rich` issues: Implementation is under `rich/`.
- Read only the specific target files and lines using `read_file` or search tools. Do not wander across unrelated files.
- If the problem statement does not provide explicit file paths, search for its most specific identifier or error string with `git grep -n "text" -- '*.py' | head -20`.
- Do not use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`: they only accept exact fully-qualified ids and usually return nothing.

### 2. Implement the Solution Directly
- Apply the minimal necessary fix or feature directly to the source files using `edit_file` or `write_file`.
- Strictly adhere to specified error strings, exception types, HTTP status codes, and API signatures.
- **For Features**: Find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern, including registration points like `__all__`, option lists, dispatch tables, and imports.
- **For Documentation Code Tasks** (e.g. FastAPI): Edit executable code under `docs_src/` (tests directly import and run examples from `docs_src/`).
- **CRITICAL FOR `edit_file`**: Ensure `old_string` includes 2–3 lines of surrounding unique context so it matches uniquely. Copy indentation verbatim from `read_file`.

### 3. Run Targeted Tests Only (Existing Tests May Be Broken)
- **Run ONLY Targeted Tests**: Run only the specific test file or test method directly verifying the bug or feature you modified (e.g. `timeout 150 pytest tests/test_target.py -k test_feature` or `python3 -m unittest tests.test_target`).
- **Be Aware That Existing Tests May Be Broken**: Many repositories contain pre-existing test breakages, missing test data fixtures (e.g. `/test_data`), or environment import errors unrelated to your task.
- **Do NOT Attempt to Fix Existing Tests**: If an existing test fails due to pre-existing repository issues or missing fixtures, IGNORE IT. Never spend turns attempting to repair pre-existing test failures, create test stubs, or alter test code.
- **STRICT RULE: NEVER Run Bare Pytest or Full-Repo Sweeps**: NEVER run bare `pytest`, `pytest .`, `python3 -m unittest discover`, or full-repo test suites without specifying a target file. Full test suites take several minutes, cause catastrophic timeouts, and exhaust your turn and time budgets.
- If you need to locate the test file, find it explicitly with `find tests -name "*<name>*.py"` instead of running the test runner across the repo.
- If no test can run, verify syntax with `python3 -m py_compile <file>`.

### 4. Immediate Patch Submission
- Once your targeted test passes:
  1. Confirm no test files were touched. If any file in `tests/` was modified, revert it: `git checkout -- tests/`.
  2. Call `submit_patch` immediately.
  3. Verify `patch_size > 0` and `files_changed > 0`.
  4. Output a short summary of the fix to end the session.

## Budget: 28 tool calls, 8 minutes
- You have at most 28 tool calls and 8 minutes for this task. When either runs out, the working tree is graded as it is, so an edit made in time counts even if you never reach `submit_patch`.
- Use at most about 10 tool calls to find and read the code.
- By your 12th tool call you must have edited a source file. If you have not, stop exploring and make your best-guess edit in the most likely place now. Runs that are still only reading after 12 calls almost never succeed; more reading does not help.
- After your edit: one compile check or one targeted test, a correction if needed, then `submit_patch`. Do not start new exploration after your 20th tool call.

## Anti-Patterns to Avoid
- **NEVER modify, create, or delete test files** (`*_test.py`, `test_*.py`, or anything under `tests/`). All changes must be to source implementation files. Modifying tests results in an automatic evaluation failure.
- **NEVER run full repository test suites** (e.g., bare `pytest` or `pytest .`) — always specify the exact test file path.
- **NEVER attempt to fix or repair existing tests or pre-existing repository breakages** — your task is strictly to implement the fix for the reported issue in source code.
- **NEVER search outside `/workspace`** for source files or packages (e.g., `/usr/local/lib/`, `/wheels/`, `/opt/`). All repository code and test dependencies are pre-installed. If `ModuleNotFoundError` occurs during test runs, focus on fixing code under `/workspace`, not looking for missing system packages.
- Scratch files go to `/tmp` only. Anything left in `/workspace` becomes part of your patch and will fail evaluation.
- Do NOT spend turns running broad exploratory searches if the file path or symbol is obvious.
- Do NOT refactor or reformat unrelated functions or files.
- Do NOT conclude without submitting a non-empty patch (`patch_size > 0`). Every task requires concrete source modifications. Concluding that the codebase is already clean without making changes is an anti-pattern."""

AGENT_FILES = {
  "agent.yaml": 


## 3. Package and Validate `submission.zip` (Guaranteed Early Execution)

We package `submission.zip` immediately and validate it against all Google ADK and competition constraints.
This guarantees that `submission.zip` exists in `/kaggle/working/` regardless of subsequent cell execution.


In [ ]:
import zipfile
from pathlib import Path

# Package all files in AGENT_DIR into submission.zip
ZIP_OUTPUT_PATH.unlink(missing_ok=True)
with zipfile.ZipFile(ZIP_OUTPUT_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(AGENT_DIR.rglob('*')):
        if file_path.is_file():
            arcname = file_path.relative_to(AGENT_DIR).as_posix()
            zf.write(file_path, arcname)

# Comprehensive Pre-Flight Validation
with zipfile.ZipFile(ZIP_OUTPUT_PATH, 'r') as zf:
    names = zf.namelist()
    infos = zf.infolist()
    total_unpacked = sum(i.file_size for i in infos)
    zip_bytes = ZIP_OUTPUT_PATH.read_bytes()
    sha256 = hashlib.sha256(zip_bytes).hexdigest()

    # Rule 1: Exactly one root agent.yaml
    assert 'agent.yaml' in names, 'Missing root agent.yaml!'
    assert names.count('agent.yaml') == 1, 'Multiple root agent.yaml!'
    assert 'eval_config.yaml' in names, 'Missing eval_config.yaml!'
    assert 'configs/sampling.yaml' in names, 'Missing configs/sampling.yaml!'
    assert 'prompts/system.md' in names, 'Missing prompts/system.md!'

    # Rule 2: Allowed file extensions only
    ALLOWED_EXTENSIONS = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
    for n in names:
        ext = Path(n).suffix.lower()
        assert ext in ALLOWED_EXTENSIONS, f'Disallowed extension: {n}'

    # Rule 3: Size limit (< 3 GiB)
    assert total_unpacked < 3 * 1024 * 1024 * 1024, 'Archive exceeds 3 GiB limit!'

    # Rule 4: Clean model declaration
    assert 'model: gemma-4-31b-it-qat-w4a16-ct' in AGENT_FILES['agent.yaml']

print(f'[SUCCESS] Packaged {ZIP_OUTPUT_PATH.name} ({len(zip_bytes):,} bytes, SHA-256: {sha256[:16]}...)')
print(f'Archive members ({len(names)} files):')
for n in sorted(names):
    print(f'  - {n}')


## 4. Code Graph Intelligence & AST Exploration (Optional / Guarded)

Demonstrates code graph inspection on available repositories without raising unhandled exceptions on hidden test sets.


In [ ]:
# Native inspection of pre-computed graphs without third-party dependencies
GRAPH_DIR = DATA_DIR / 'graphs'
EMBEDDINGS_DIR = DATA_DIR / 'embeddings'

if tasks and GRAPH_DIR.exists():
    try:
        sample_task = tasks[0]
        repo_short = sample_task.get('repo', '').split('/')[-1]
        commit = sample_task.get('base_commit', '')
        inst_id = sample_task.get('instance_id', '')
        
        # Check commit-named or task-named graph files using standard library json
        graph_file = None
        for stem in [f"{repo_short}_{commit}", inst_id]:
            p = GRAPH_DIR / f"{stem}.json"
            if p.is_file() and p.stat().st_size > 100:
                graph_file = p
                break
        
        if graph_file:
            data = json.loads(graph_file.read_text(encoding='utf-8-sig'))
            nodes = data.get('nodes', [])
            edges = data.get('edges', data.get('links', []))
            print(f'[+] AST graph loaded natively for {repo_short}: {len(nodes):,} nodes, {len(edges):,} edges ({graph_file.name})')
        else:
            print('[+] AST graph assets verified (pre-computed graphs ready for evaluation cluster).')
    except Exception as e:
        print(f'[*] AST graph exploration note: {e}')
else:
    print('[+] Graph assets: will be loaded inside evaluation sandbox during competition scoring.')


## 5. Local Inference & Evaluation Policy

During Kaggle submission, evaluation runs asynchronously on Kaggle's backend evaluation cluster using `submission.zip`.
Local notebook vLLM execution is guarded and skipped during automated notebook submission to prevent timeouts.


In [ ]:
# Execution flags
# Set to True only when running an interactive GPU smoke test in the notebook editor
ENABLE_LOCAL_VLLM_SMOKE_TEST = False

if ENABLE_LOCAL_VLLM_SMOKE_TEST:
    try:
        import torch
        from adk_submission import VllmConfig, VllmServer
        print('[+] Interactive local smoke test enabled.')
    except Exception as e:
        print(f'[*] Local vLLM import note: {e}')
else:
    print('[+] Standalone submission mode active.')
    print('    Evaluation is handled automatically by Kaggle using /kaggle/working/submission.zip.')


## 6. Enterprise Release Gatekeeper & Verification Summary

Verifies that `submission.zip` is ready for final scoring.


In [ ]:
# Final confirmation of submission artifact
assert ZIP_OUTPUT_PATH.exists(), 'ERROR: submission.zip was not found!'
zip_stat = ZIP_OUTPUT_PATH.stat()
assert zip_stat.st_size > 0, 'ERROR: submission.zip is empty!'

release_summary = {
    'archive': ZIP_OUTPUT_PATH.name,
    'size_bytes': zip_stat.st_size,
    'sha256': hashlib.sha256(ZIP_OUTPUT_PATH.read_bytes()).hexdigest(),
    'files_count': len(AGENT_FILES),
    'architecture': 'HADL Dual-Loop Unified Agent (System 1 + System 2 Deliberation)',
    'base_model': 'gemma-4-31b-it-qat-w4a16-ct',
    'eval_config': {
        'timeout_seconds': 240,
        'max_tool_calls': 28,
        'max_time_minutes': 8,
        'max_turns': 80
    },
    'calibration': '0.18 - 0.24+ Leaderboard Target',
    'status': 'READY_FOR_KAGGLE_LEADERBOARD',
}

print(json.dumps(release_summary, indent=2))

hud_badge = r'''
+==============================================================================+
|              HADL DUAL-LOOP UNIFIED CHAMPIONSHIP AGENT                       |
|        Calibrated 0.18 Baseline + Synergistic /plan /boost /goal             |
+==============================================================================+
|  [OK] 1. Cognitive Architecture: HADL DUAL-LOOP SINGLE-AGENT SYSTEM 1 & 2    |
|  [OK] 2. Timeout Boost         : 240s HARNESS TIMEOUT (NO requests FALSE FAIL)|
|  [OK] 3. Time & Call Budget    : 8 MIN / 28 CALLS / 80 TURNS (THE 12-CALL INV)|
|  [OK] 4. vLLM Protection       : 8K MAX TOKENS (PREVENTS 32K CONTEXT CRASH)   |
|  [OK] 5. Thinking Budget       : 4096 BUDGET + THOUGHTS ENABLED               |
|  [OK] 6. Cognitive Triad       : /goal -> /plan -> /boost -> /goal AUDIT      |
+==============================================================================+
|        >>> STATUS: 100% AIR-GAPPED, CRASH-PROOF & READY FOR LEADERBOARD <<<  |
+==============================================================================+
'''
print(hud_badge)
